### STEP : PICRUST2 Analysis



#### Example

- [PICRUST2 tutorial](https://github.com/picrust/picrust2/wiki/q2-picrust2-Tutorial)
- [Limitations](https://github.com/picrust/picrust2/wiki/Key-Limitations)


#### Methods
- [composition](https://docs.qiime2.org/2022.8/plugins/available/composition/)

## Setup and settings

In [ ]:
# Importing packages
from utils import setup_temp_environment

env = setup_temp_environment("/mnt/data/tmp")

import os

import biom
import pandas as pd
from biom import Table
from biom.util import biom_open
from qiime2 import Artifact, Metadata, Visualization
from qiime2.plugins.diversity.pipelines import core_metrics
from qiime2.plugins.feature_table.methods import filter_samples, filter_seqs
from qiime2.plugins.feature_table.pipelines import summarize

%matplotlib inline

### Receiving the parameters

The following cell can receive parameters using the [papermill](https://papermill.readthedocs.io/en/latest/) tool.

In [ ]:
metadata_file = "/home/lauro/nupeb/rede-micro/redemicro-miliane-nutri/data/raw/metadata/miliane-metadata-CxAC.tsv"
base_dir = os.path.join(
    "/", "home", "lauro", "nupeb", "rede-micro", "redemicro-miliane-nutri"
)
experiment_name = "miliane-CxAC-trim"
class_col = "group-id"
replace_files = False

In [ ]:
# Parameters
experiment_name = "experiment-test"
base_dir = "/mnt/data/qiime2-experiments"
manifest_file = "manifest.csv"
metadata_file = "metadata.tsv"
class_col = "group"
classifier_file = "qiime2-classifiers/silva-138.2-v3v4-341f-806r-nb-classifier.qza"
read_layout = "paired-end"
replace_files = False
phred = 20
trunc_f = 0
trunc_r = 0
overlap = 12
threads = 8
trim = {
    "overlap": 8,
    "forward_primer": "CCTACGGGRSGCAGCAG",
    "reverse_primer": "GGACTACHVGGGTWTCTAAT",
}

In [ ]:
experiment_folder = os.path.abspath(
    os.path.join(base_dir, "experiments", experiment_name)
)
img_folder = os.path.abspath(os.path.join(experiment_folder, "imgs"))

### Defining names, paths and flags

In [ ]:
# QIIME2 Artifacts folder
qiime_folder = os.path.join(experiment_folder, "qiime-artifacts")

# Input - DADA2 Artifacts
dada2_tabs_path = os.path.join(qiime_folder, "dada2-tabs.qza")
dada2_reqs_path = os.path.join(qiime_folder, "dada2-reps.qza")

# PICRUST2 folder
picrust2_folder = os.path.abspath(os.path.join(experiment_folder, "picrust2"))

# Create path if it not exist
if not os.path.isdir(picrust2_folder):
    os.makedirs(picrust2_folder)
    print(f"New picrust2-artifacts folder path created: {picrust2_folder}")

In [ ]:
# Define paths for metagenome predicted function abundance tables
ec_fpath = os.path.join(picrust2_folder, "ec-pred-metagen.tsv")
ko_fpath = os.path.join(picrust2_folder, "ko-pred-metagen.tsv")
pathway_fpath = os.path.join(picrust2_folder, "pathway-abundance.tsv")

# Define paths for metagenome predicted function abundance tables - with descriptions
ec_desc_fpath = os.path.join(picrust2_folder, "ec-desc-pred-metagen.tsv")
ko_desc_fpath = os.path.join(picrust2_folder, "ko-desc-pred-metagen.tsv")
pathway_desc_fpath = os.path.join(picrust2_folder, "pathway-desc-pred-metagen.tsv")


# Define paths for metagenome function artifacts
ec_path = os.path.join(picrust2_folder, "ec-pred-metagen.qza")
ko_path = os.path.join(picrust2_folder, "ko-pred-metagen.qza")
pathway_path = os.path.join(picrust2_folder, "pathway-abundance.qza")

# Define paths for metagenome function visualization artifacts
ec_viz_path = os.path.join(picrust2_folder, "ec-pred-metagen.qzv")
ko_viz_path = os.path.join(picrust2_folder, "ko-pred-metagen.qzv")
pathway_viz_path = os.path.join(picrust2_folder, "pathway-abundance.qzv")

In [ ]:
seq_file = os.path.join(picrust2_folder, "dna-sequences.fasta")
biom_file = os.path.join(picrust2_folder, "feature-table.biom")
tmp_folder = os.path.join(picrust2_folder, "picrust2_tmp")

## Step execution

### Load input files

This Step import the QIIME2 `FeatureTable[Frequency]` Artifact and the `Metadata` file.

In [ ]:
# Load Metadata
metadata_qa = Metadata.load(metadata_file)

# Load FeatureTable[Frequency]
tabs = Artifact.load(dada2_tabs_path)

# Load FeatureTable[Sequence]
seqs = Artifact.load(dada2_reqs_path)

In [ ]:
# Filter FeatureTable[Frequency | RelativeFrequency | PresenceAbsence | Composition] based on Metadata sample ID values
tabs = filter_samples(
    table=tabs,
    metadata=metadata_qa,
).filtered_table
# Filter SampleData[SequencesWithQuality | PairedEndSequencesWithQuality | JoinedSequencesWithQuality] based on Metadata sample ID values; returns FeatureData[Sequence | AlignedSequence]
seqs = filter_seqs(
    data=seqs,
    table=tabs,
).filtered_data

In [ ]:
seqs.export_data(output_dir=picrust2_folder)
table = tabs.view(biom.Table)
with biom_open(str(biom_file), "w") as f:
    table.to_hdf5(f, "QIIME2")

### Execute full pipeline

The entire PICRUSt2 pipeline will be run using a single script, called `picrust2_pipeline.py` through Docker. This method will run each of the 4 key steps:

1. sequence placement
2. hidden-state prediction of genomes
3. metagenome prediction
4. pathway-level predictions.

More information on:
- [Documentation](https://github.com/picrust/picrust2/wiki/Full-pipeline-script).
- [Standard PICRUSt2 Output](https://github.com/picrust/picrust2/wiki/Standard-PICRUSt2-output)

After the pipeline is complete, the predicted metagenome and pathway abundance tables will be extracted and copied to the PICRUSt2 folder. Finally, a description column will be added to each of the predicted files.

In [ ]:
# Execute the PICRUSt2 pipeline using Docker
cmd = f"picrust2_pipeline.py -s {os.path.basename(seq_file)} -i {os.path.basename(biom_file)} -o {os.path.basename(tmp_folder)} -p {threads}"
!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 /bin/bash -c "{cmd}"

In [ ]:
# Extract the predicted metagenome and pathway abundance tables
!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 gunzip {os.path.basename(tmp_folder)}/EC_metagenome_out/pred_metagenome_unstrat.tsv.gz

!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 gunzip {os.path.basename(tmp_folder)}/KO_metagenome_out/pred_metagenome_unstrat.tsv.gz

!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 gunzip {os.path.basename(tmp_folder)}/pathways_out/path_abun_unstrat.tsv.gz

In [ ]:
# Copy the predicted metagenome and pathway abundance tables to the PICRUSt2 folder
!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 cp {os.path.basename(tmp_folder)}/EC_metagenome_out/pred_metagenome_unstrat.tsv {os.path.basename(ec_fpath)}

!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 cp {os.path.basename(tmp_folder)}/KO_metagenome_out/pred_metagenome_unstrat.tsv {os.path.basename(ko_fpath)}

!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 cp {os.path.basename(tmp_folder)}/pathways_out/path_abun_unstrat.tsv {os.path.basename(pathway_fpath)}

In [ ]:
# Remove temporary folder
!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 rm -rf {os.path.basename(tmp_folder)}

In [ ]:
# Add description column to predicted files
!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 add_descriptions.py -i {os.path.basename(ec_fpath)} -m EC -o {os.path.basename(ec_desc_fpath)}

!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 add_descriptions.py -i {os.path.basename(ko_fpath)} -m KO -o {os.path.basename(ko_desc_fpath)}

!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 add_descriptions.py -i {os.path.basename(pathway_fpath)} -m METACYC -o {os.path.basename(pathway_desc_fpath)}

In [ ]:
# Change ownership of the files to the current user
!docker run --rm --workdir /data -v {picrust2_folder}:/data quay.io/biocontainers/picrust2:2.6.3--pyhdfd78af_2 chown -R $(id -u):$(id -g) /data

# Import predicted metagenome and pathway abundance tables as QIIME2 Artifacts

In [ ]:
_col = "description"
for f in (ec_desc_fpath, ko_desc_fpath, pathway_desc_fpath):

    if not f.endswith(".tsv"):
        print(f"Invalid file (not TSV): {f}")
        continue

    # 1. Load the TSV file using pandas
    # Assumes the first column holds the Feature ID / OTU ID
    df = pd.read_csv(f, sep="\t", index_col=0)

    # 2. Separate metadata if it exists in your TSV
    if _col in df.columns:
        obs_metadata = [{_col: [tax]} for tax in df[_col]]
        # Drop the taxonomy text column so only numerical data remains
        df = df.drop(columns=[_col])
    else:
        obs_metadata = None

    # 3. Extract IDs and matrix data
    sample_ids = list(df.columns)
    observation_ids = list(df.index.astype(str))
    data_matrix = df.values

    # 4. Construct the BIOM Table object
    biom_table = Table(
        data_matrix, observation_ids, sample_ids, observation_metadata=obs_metadata
    )

    # 5. Write out the BIOM file in HDF5 format
    new_name = f.replace(".tsv", ".biom")
    with biom_open(new_name, "w") as f:
        biom_table.to_hdf5(f, generated_by="Python TSV Converter")

In [ ]:
# Load predicted metagenome and pathway abundance tables as QIIME2 Artifacts
ec_metagenome = Artifact.import_data(
    "FeatureTable[Frequency]", ec_desc_fpath.replace(".tsv", ".biom")
)
ko_metagenome = Artifact.import_data(
    "FeatureTable[Frequency]", ko_desc_fpath.replace(".tsv", ".biom")
)
pathway_abundance = Artifact.import_data(
    "FeatureTable[Frequency]", pathway_desc_fpath.replace(".tsv", ".biom")
)

### Persist created artifacts

We will define file paths and persist all artifacts. We start with `.qza` files. We will save the visualization files in sequence as `qzv` files. Finally, we save a `biom`-like file as `tsv` with brief descriptions of all functions.

In [ ]:
# Export artifact folder
ec_metagenome.export_data(output_dir=ec_path.split(".")[0])
ko_metagenome.export_data(output_dir=ko_path.split(".")[0])
pathway_abundance.export_data(output_dir=pathway_path.split(".")[0])

# Save artifacts as .qza files
ec_metagenome.save(ec_path)
ko_metagenome.save(ko_path)
pathway_abundance.save(pathway_path)

In [ ]:
need_viz = replace_files
need_viz |= not (
    os.path.isfile(ec_viz_path)
    and os.path.isfile(ko_viz_path)
    and os.path.isfile(pathway_viz_path)
)
if need_viz:
    # Create visualization artifacts
    ec_viz = summarize(table=ec_metagenome, metadata=metadata_qa)
    ko_viz = summarize(table=ko_metagenome, metadata=metadata_qa)
    path_viz = summarize(table=pathway_abundance, metadata=metadata_qa)

    # Save visualization artifacts as .qzv files
    ec_viz.summary.save(ec_viz_path)
    ko_viz.summary.save(ko_viz_path)
    path_viz.summary.save(pathway_viz_path)

In [ ]:
tables = (ec_metagenome, ko_metagenome, pathway_abundance)
table_types = ("ec", "ko", "path")

for i, t in enumerate(tables):
    print(f"Processing {table_types[i]}")
    results = core_metrics(
        table=t,
        sampling_depth=1,
        metadata=metadata_qa,
        n_jobs=6,
    )
    emperor_out = os.path.join(
        picrust2_folder, f"{table_types[i]}-bray_curtis_emperor.qzv"
    )
    results.bray_curtis_emperor.save(emperor_out)
    emperor_out = os.path.join(picrust2_folder, f"{table_types[i]}-jaccard_emperor.qzv")
    results.jaccard_emperor.save(emperor_out)

### Aggregate by sum groups counts

We create new columns with the aggregation of the counts for all groups.

In [ ]:
# Create a new DataFrame with Metadata info
metadata_df = metadata_qa.to_dataframe()
# Select all groups
group_ids = metadata_df[class_col].unique()

In [ ]:
# Iterate over all PICRUST2 files
for f in (ec_desc_fpath, ko_desc_fpath, pathway_desc_fpath):
    # Create new DataFrame to store new data
    df = pd.read_csv(f, sep="\t")
    new_df = pd.DataFrame()
    # Iterate over all groups
    for g in group_ids:
        # Select all IDs from current group
        idx = metadata_df[metadata_df[class_col] == g].index
        # Sum all values from samples of the current group
        tmp = df.loc[:, idx].sum(axis=1)
        # Create a new column with the sum
        new_df[g] = tmp
    # Sum the values of all groups
    new_df["Total"] = new_df.sum(axis=1)
    # Join new columns into the end of the original DataFrame
    new_df = pd.concat([df, new_df])
    # Write joined DataFrame to file - overwrite
    new_df.to_csv(f, index=False, sep="\t")